# ElliGAT — Journal-Correction & Reproducibility Notebook

This notebook applies the research-validity corrections to the Bitcoin Transaction Fraud Detection repository, runs preflight checks, then reruns the full experiment **one seed at a time** so completed seeds remain on Google Drive if Colab disconnects.

## What this notebook changes

- Selects operating thresholds on validation predictions, never on test labels.
- Adds PR-AUC and confusion-matrix counts to evaluation output.
- Splits on complete timesteps and directs cross-timestep graph messages from earlier to later timesteps.
- Restricts pseudo-label propagation to unlabelled nodes available by the training cutoff; skips one-class pseudo-label collapse.
- Removes fabricated wallet/amount/clock “velocity” proxies because the public Elliptic dataset does not provide those real-world fields.
- Adds preflight checks, resumable per-seed runs, result aggregation, and optional FAGA/PLP ablation stages.

**Important:** These protocol changes alter the experiment. Previous metrics are not directly comparable and must not be reused as corrected results. This notebook cannot guarantee journal acceptance or establish novelty by itself.

### Recommended runtime
1. In Colab, choose **Runtime → Change runtime type → T4 GPU** (or another supported GPU).
2. Run cells from top to bottom.
3. Keep Drive mounted; results are written to Drive per seed.
4. If disconnected, reconnect, rerun setup cells, and rerun the full-pipeline cell. Completed seeds marked `DONE.txt` are skipped.


In [1]:
# CONFIGURATION — edit only if needed
GITHUB_URL = "https://github.com/ria0304/Bitcoin-Transaction-Fraud-Detection-"
SEEDS = [42, 123, 2024, 17, 99]
EPOCHS = 300
PRETRAIN_EPOCHS = 50
PLP_EPOCHS = 50
HIDDEN_DIM = 128           # automatically retries at 64/32 if CUDA OOM occurs
LOSS_NAME = "asymmetric"  # "asymmetric" or "focal"
EDGE_ENCODING = "sinusoidal"
MC_SAMPLES = 30
EXPLAIN_NODES = 10
EXPLAIN_EPOCHS = 100

DATA_DIR = "/content/data/elliptic"
DRIVE_ROOT = "/content/drive/MyDrive/ElliGAT_Journal_Corrected"
RUN_ABLATIONS = False      # change to True only when the full five-seed run is complete
RUN_FAGA_ABLATION = False
RUN_PLP_ABLATION = False

print("Seeds:", SEEDS)
print("Epochs:", EPOCHS, "| hidden dim:", HIDDEN_DIM)
print("Drive output root:", DRIVE_ROOT)


Seeds: [42, 123, 2024, 17, 99]
Epochs: 300 | hidden dim: 128
Drive output root: /content/drive/MyDrive/ElliGAT_Journal_Corrected


## 1. Mount Drive and verify GPU

In [2]:
from google.colab import drive
import os, torch, json, time, glob, shutil, subprocess, sys, textwrap
from pathlib import Path

drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("A GPU runtime is required for the full five-seed experiment. Select Runtime → Change runtime type → GPU.")


Mounted at /content/drive
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


## 2. Clone the repository and install dependencies

In [3]:
REPO_DIR = "/content/elligat"
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
clone = subprocess.run(["git", "clone", GITHUB_URL, REPO_DIR], text=True, capture_output=True)
print(clone.stdout)
if clone.returncode:
    print(clone.stderr)
    raise RuntimeError("Repository clone failed. Check GITHUB_URL and repository access.")
%cd /content/elligat
!pip -q install torch-geometric xgboost lightgbm python-dotenv kagglehub scikit-learn pandas scipy matplotlib
import torch_geometric
print("PyG:", torch_geometric.__version__)



/content/elligat
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.2 MB/s eta 0:00:00
PyG: 2.8.1


## 3. Apply journal-protocol corrections

In [4]:
from pathlib import Path
import re, py_compile
ROOT = Path('/content/elligat')
def edit(rel, pattern, replacement, flags=re.S):
    p=ROOT/rel; s=p.read_text(); ns,n=re.subn(pattern,replacement,s,count=1,flags=flags)
    if n != 1: raise RuntimeError(f'{rel}: expected 1 match, got {n}: {pattern[:90]!r}')
    p.write_text(ns)

# Robust class mapping
edit('src/data.py', r'classes\["class"\] = classes\["class"\]\.map\(\{"1": 1, "2": 0, "unknown": np\.nan\}\)', '''raw_class = classes["class"].astype(str).str.strip().str.lower()
    classes["class"] = raw_class.map({"1": 1, "1.0": 1, "2": 0, "2.0": 0, "unknown": np.nan})''')
# Time-respecting directed edges (past -> future only, same timestep bidirectional)
edit('src/data.py', r'        src = torch\.tensor\(\[txid_to_idx\[t\] for t in el\["txId1"\]\], dtype=torch\.long\).*?\n        print\(', '''        src_np = np.asarray([txid_to_idx[t] for t in el["txId1"]], dtype=np.int64)
        dst_np = np.asarray([txid_to_idx[t] for t in el["txId2"]], dtype=np.int64)
        ts_arr = all_nodes["timestep"].to_numpy(dtype=np.int64)
        t_src, t_dst = ts_arr[src_np], ts_arr[dst_np]
        # Orient cross-timestep messages from earlier to later transactions.
        # Same-timestep edges stay bidirectional; future features cannot flow back.
        earlier_src = np.where(t_src <= t_dst, src_np, dst_np)
        later_dst   = np.where(t_src <= t_dst, dst_np, src_np)
        same_time = (t_src == t_dst)
        edge_src = np.concatenate([earlier_src, src_np[same_time][::-1]])
        edge_dst = np.concatenate([later_dst, dst_np[same_time][::-1]])
        delta = np.abs(ts_arr[edge_src] - ts_arr[edge_dst]).astype(np.float32)
        src = torch.as_tensor(edge_src, dtype=torch.long)
        dst = torch.as_tensor(edge_dst, dtype=torch.long)
        edge_index = torch.stack([src, dst], dim=0)
        edge_attr = torch.as_tensor(delta, dtype=torch.float32).unsqueeze(1)

        print(''')
# Remove fabricated proxies and use real timestep sort only
edit('src/data.py', r'VELOCITY_COLS = \[.*?\]', '''# Synthetic wallet/amount/clock proxies are excluded: the public Elliptic table
# does not contain real wallet IDs, transaction amounts, or wall-clock timestamps.
VELOCITY_COLS = []''')
edit('src/data.py', r'    # Synthetic wallet/amount proxies \(Elliptic doesn.t expose real wallets\).*?    labelled = labelled\.sort_values\("Timestamp"\)\.reset_index\(drop=True\)', '''    # Keep the dataset's ordinal timestep; do not fabricate wallet IDs, amounts,
    # or wall-clock timestamps from unrelated columns.
    labelled = labelled.sort_values(["timestep", "txId"], kind="mergesort").reset_index(drop=True)''')
edit('src/data.py', r'def add_velocity_features\(df: pd\.DataFrame\) -> pd\.DataFrame:.*?\n\n# ─── Chronological split', '''def add_velocity_features(df: pd.DataFrame) -> pd.DataFrame:
    """No-op: exclude synthetic velocity proxies unavailable in Elliptic."""
    return df.copy()


# ─── Chronological split''')
edit('src/data.py', r'def chronological_split\(df: pd\.DataFrame, train_frac=0\.70, val_frac=0\.15\):.*?\n\n# ─── Graph construction', '''def chronological_split(df: pd.DataFrame, train_frac=0.70, val_frac=0.15):
    """Split on whole timesteps so one timestep cannot straddle partitions."""
    if not df["timestep"].is_monotonic_increasing:
        df = df.sort_values(["timestep", "txId"], kind="mergesort").reset_index(drop=True)
    unique_ts = np.sort(df["timestep"].unique())
    if len(unique_ts) < 3:
        raise ValueError("At least three distinct timesteps are required for train/val/test.")
    train_ts_end = max(1, int(len(unique_ts) * train_frac))
    val_ts_end = min(len(unique_ts) - 1, train_ts_end + max(1, int(len(unique_ts) * val_frac)))
    train_cutoff = unique_ts[train_ts_end - 1]
    val_cutoff = unique_ts[val_ts_end - 1]
    ts = df["timestep"].to_numpy()
    train_end = int(np.searchsorted(ts, train_cutoff, side="right"))
    val_end = int(np.searchsorted(ts, val_cutoff, side="right"))
    return (df.iloc[:train_end].copy(), df.iloc[train_end:val_end].copy(),
            df.iloc[val_end:].copy(), train_end, val_end - train_end)


# ─── Graph construction''')
edit('src/data.py', r'    feat_cols: list,\n\) -> Data:', '''    feat_cols: list,
    all_nodes: pd.DataFrame | None = None,
) -> Data:''')
edit('src/data.py', r'    # Velocity features — scaled on training set, zero-padded elsewhere.*?\n    # Labels', '''    # Use original Elliptic features only; no synthetic proxy features.
    node_feats = X_scaled
    x = torch.as_tensor(node_feats, dtype=torch.float32)

    # Labels''')
edit('src/data.py', r'    data = Data\(\n        x=x,.*?\n    \)\n    return data, scaler_raw, scaler_v', '''    if all_nodes is not None:
        node_timestep_np = all_nodes["timestep"].to_numpy(dtype=np.int64)
    else:
        node_timestep_np = np.full(N_all, -1, dtype=np.int64)
        for row_i, global_i in enumerate(labelled_global_idx.numpy()):
            node_timestep_np[global_i] = int(df.iloc[row_i]["timestep"])
    node_timestep = torch.as_tensor(node_timestep_np, dtype=torch.long)
    train_cutoff = int(df.iloc[train_size - 1]["timestep"]) if train_size else -1
    plp_eligible = (labels_full == -1) & (node_timestep <= train_cutoff)
    data = Data(
        x=x, edge_index=edge_index, edge_attr=edge_attr, y=labels_full,
        train_mask=tr_mask, val_mask=vl_mask, test_mask=te_mask,
        node_timestep=node_timestep, plp_eligible_mask=plp_eligible,
    )
    return data, scaler_raw, None''')
# Update edge log wording
p=ROOT/'src/data.py'; s=p.read_text().replace('f"{edge_index.shape[1]} undirected"','f"{edge_index.shape[1]} time-respecting directed"'); p.write_text(s)

# Threshold safety, metrics
edit('src/trainer.py', r'    matthews_corrcoef,\n\)', '''    matthews_corrcoef,
    average_precision_score,
    confusion_matrix,
)''')
edit('src/trainer.py', r'def compute_metrics\(\n    y_true: np\.ndarray,\n    probs: np\.ndarray,\n    threshold: float \| None = None,\n\) -> dict:\n    if threshold is None:\n        threshold = find_best_threshold\(y_true, probs\)', '''def compute_metrics(
    y_true: np.ndarray,
    probs: np.ndarray,
    threshold: float | None = 0.5,
) -> dict:
    """Use a frozen threshold; callers must select thresholds on validation only."""
    if threshold is None:
        threshold = 0.5''')
edit('src/trainer.py', r'        "ROC-AUC":\s+float\(roc_auc_score\(y_true, probs\)\),\n        "threshold":', '''        "ROC-AUC":      float(roc_auc_score(y_true, probs)) if len(np.unique(y_true)) > 1 else float("nan"),
        "PR-AUC":       float(average_precision_score(y_true, probs)) if len(np.unique(y_true)) > 1 else float("nan"),
        "TN":           int(confusion_matrix(y_true, y_pred, labels=[0, 1])[0, 0]),
        "FP":           int(confusion_matrix(y_true, y_pred, labels=[0, 1])[0, 1]),
        "FN":           int(confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 0]),
        "TP":           int(confusion_matrix(y_true, y_pred, labels=[0, 1])[1, 1]),
        "threshold":''')
edit('src/trainer.py', r'    val_probs = probs_mc_val\.mean\(axis=0\)\n    val_unc\s+= probs_mc_val\.std\(axis=0\)\n    val_y\s+= data\.y\[data\.val_mask\]\.cpu\(\)\.numpy\(\)\n', '''    val_probs = probs_mc_val.mean(axis=0)
    val_unc   = probs_mc_val.std(axis=0)
    val_y     = data.y[data.val_mask].cpu().numpy()
    decision_threshold = find_best_threshold(val_y, val_probs) if len(np.unique(val_y)) > 1 else 0.5
''')
edit('src/trainer.py', r'        result = evaluate_augt\(test_y, test_probs, test_probs, test_unc,\s*base_low, base_high, k\)', '''        result = evaluate_augt(
            test_y, test_probs, test_probs, test_unc,
            base_low, base_high, k, threshold=decision_threshold
        )''')
edit('src/trainer.py', r'        # Fallback: standard evaluation\n        result = compute_metrics\(test_y, test_probs\)', '''        # Test labels are never used to tune the operating threshold.
        result = compute_metrics(test_y, test_probs, threshold=decision_threshold)''')

# AUGT metrics follow same threshold and expose PR-AUC / confusion counts
edit('src/uncertainty.py', r'    balanced_accuracy_score,\n\)', '''    balanced_accuracy_score,
    average_precision_score,
    confusion_matrix,
)''')
edit('src/uncertainty.py', r'    uncertainty: np\.ndarray,\n    n_thresholds: int = 100,\n\)', '''    uncertainty: np.ndarray,
    n_thresholds: int = 100,
    decision_threshold: float = 0.5,
)''')
edit('src/uncertainty.py', r'y_pred = \(probs\[retain\] >= 0\.5\)\.astype\(int\)', 'y_pred = (probs[retain] >= decision_threshold).astype(int)')
edit('src/uncertainty.py', r'    k: float,\n\) -> dict:\n    """\n    Full evaluation under the uncertainty-aware AUGT decision rule\.', '''    k: float,
    threshold: float = 0.5,
) -> dict:
    """
    Full evaluation under the uncertainty-aware AUGT decision rule.''')
edit('src/uncertainty.py', r'    # ── Standard metrics on all nodes \(threshold=0\.5\) ────────────────────\n    pred_all = \(probs >= 0\.5\)\.astype\(int\)', '''    # Standard metrics use the threshold selected on validation data.
    pred_all = (probs >= threshold).astype(int)''')
edit('src/uncertainty.py', r'aurc, coverages, risks = compute_aurc\(y_true, probs, unc_std\)', '''aurc, coverages, risks = compute_aurc(y_true, probs, unc_std, decision_threshold=threshold)''')
edit('src/uncertainty.py', r'"ROC-AUC":\s+auc_all,\n        "F1":', '''"ROC-AUC":             auc_all,
        "PR-AUC":              float(average_precision_score(y_true, probs)) if len(np.unique(y_true)) > 1 else float("nan"),
        "TN":                  int(confusion_matrix(y_true, pred_all, labels=[0, 1])[0, 0]),
        "FP":                  int(confusion_matrix(y_true, pred_all, labels=[0, 1])[0, 1]),
        "FN":                  int(confusion_matrix(y_true, pred_all, labels=[0, 1])[1, 0]),
        "TP":                  int(confusion_matrix(y_true, pred_all, labels=[0, 1])[1, 1]),
        "F1":''')
edit('src/uncertainty.py', r'"threshold":\s+0\.5,', '"threshold":           float(threshold),')

# PLP restricts to training-period unlabelled nodes and prevents class collapse
edit('src/pseudo_label.py', r'unlabelled = \(data\.y == -1\)', 'unlabelled = getattr(data, "plp_eligible_mask", (data.y == -1))')
edit('src/pseudo_label.py', r'    print\(f"  \[PLP\] Phase-3 re-training \(\{epochs\} epochs, lambda_pl=\{lambda_pl\}\) …"\)\n    pseudo_label_finetune\(', '''    pseudo_classes = torch.unique(pseudo_y[pseudo_mask])
    if pseudo_classes.numel() < 2:
        cls = int(pseudo_classes.item())
        label_name = "fraud" if cls == 1 else "licit"
        print(f"  [PLP] WARNING: only {label_name} pseudo-labels selected; skipping phase 3.")
        return pseudo_mask, pseudo_y

    print(f"  [PLP] Phase-3 re-training ({epochs} epochs, lambda_pl={lambda_pl}) …")
    pseudo_label_finetune(''')

# Validation thresholds for tabular models and MetaEnsemble
edit('src/baselines.py', r'from src\.trainer import compute_metrics', 'from src.trainer import compute_metrics, find_best_threshold')
edit('src/baselines.py', r'        self\.models: dict = \{\}', '        self.models: dict = {}\n        self.thresholds: dict[str, float] = {}')
edit('src/baselines.py', r'        rf\.fit\(X_train, y_train\)\n        self\.models\["RandomForest"\] = rf', '''        rf.fit(X_train, y_train)
        self.models["RandomForest"] = rf
        self.thresholds = {
            name: find_best_threshold(np.asarray(y_val), probs)
            for name, probs in self.predict_proba(X_val).items()
        }''')
edit('src/baselines.py', r'            results\[name\] = compute_metrics\(y_test, probs\)', '            results[name] = compute_metrics(y_test, probs, threshold=self.thresholds.get(name, 0.5))')
edit('src/baselines.py', r'        self\._fitted = False', '        self._fitted = False\n        self.threshold = 0.5')
edit('src/baselines.py', r'        self\.meta\.fit\(X_meta, y_val\)\n        self\._fitted = True', '''        self.meta.fit(X_meta, y_val)
        val_meta_probs = self.meta.predict_proba(X_meta)[:, 1]
        self.threshold = find_best_threshold(np.asarray(y_val), val_meta_probs)
        self._fitted = True''')
edit('src/baselines.py', r'        result = compute_metrics\(y_test, probs\)', '        result = compute_metrics(y_test, probs, threshold=self.threshold)')

# Pass all-node timesteps into graph builder for PLP eligibility.
for rel in ['main.py','run_clean_reproduction.py']:
    p=ROOT/rel; s=p.read_text()
    old='''        train_size, val_size, feat_cols,
    )'''
    if old in s: s=s.replace(old, '''        train_size, val_size, feat_cols, all_nodes=all_nodes,
    )''', 1)
    else:
        old='''    train_size, val_size, feat_cols,
)'''
        if old not in s: raise RuntimeError(f'{rel}: build_graph call not found')
        s=s.replace(old, '''    train_size, val_size, feat_cols, all_nodes=all_nodes,
)''', 1)
    p.write_text(s)
for rel in ['run_faga_ablation.py','run_plp_ablation.py']:
    p=ROOT/rel; s=p.read_text(); old='build_graph(df, X_all, edge_index, edge_attr, txid_to_idx, train_size, val_size, feat_cols)'
    if old not in s: raise RuntimeError(f'{rel}: build_graph call not found')
    p.write_text(s.replace(old, 'build_graph(df, X_all, edge_index, edge_attr, txid_to_idx, train_size, val_size, feat_cols, all_nodes=all_nodes)', 1))

# Keep documentation/log labels aligned with the corrected feature protocol.
p=ROOT/"src/data.py"; s=p.read_text()
s=s.replace("# ─── Build directed + undirected edge index ──────────────────────────────", "# ─── Build time-respecting directed edge index ───────────────────────────")
s=s.replace("Node features (N × 173):\n        166 raw Elliptic features\n        +7  velocity features (zero-padded for unlabelled nodes)", "Node features: standardized original Elliptic transaction features only")
p.write_text(s)
p=ROOT/"main.py"; s=p.read_text().replace("Step 2: Engineering velocity features …", "Step 2: Excluding synthetic proxies; using original Elliptic features …"); p.write_text(s)

# Preserve per-run thresholds and confusion counts in metrics.json.
edit('main.py', r'        if ea_plp is not None:', '''        out["_per_seed_metrics"] = {
            name: [
                {
                    key: (value.item() if isinstance(value, np.generic) else value)
                    for key, value in result.items()
                    if key not in _SKIP_KEYS and np.isscalar(value)
                    and not isinstance(value, (str, bytes))
                }
                for result in run_results
            ]
            for name, run_results in results_per_seed.items() if run_results
        }
        if ea_plp is not None:''')

# Include PR-AUC in cross-seed aggregate reports too.
edit('scripts/aggregate_results.py', r'    "ROC-AUC",\n    "F1",', '    "ROC-AUC",\n    "PR-AUC",\n    "F1",')

# Main aggregates PR-AUC as a headline metric.
edit('main.py', r'METRICS = \["ROC-AUC", "F1", "MCC", "Balanced Acc", "Precision", "Recall"\]', 'METRICS = ["ROC-AUC", "PR-AUC", "F1", "MCC", "Balanced Acc", "Precision", "Recall"]')
(ROOT/'JOURNAL_PROTOCOL_NOTES.md').write_text('''# Journal protocol changes applied by the Colab notebook\n\n- Thresholds are selected using validation predictions and frozen for test metrics.\n- ROC-AUC, PR-AUC, and confusion counts are reported.\n- Cross-timestep graph edges point from earlier to later timesteps; same-timestep edges are bidirectional.\n- Train/validation/test partitions use whole timesteps.\n- PLP only uses unlabelled nodes at or before the labelled training cutoff and skips one-class pseudo-label collapse.\n- Synthetic wallet/amount/clock proxies are excluded because public Elliptic data do not provide real wallet IDs, transaction amounts, or wall-clock timestamps.\n- All old results must be regenerated; this patch does not establish novelty or guarantee acceptance.\n''')
for rel in ['src/data.py','src/trainer.py','src/uncertainty.py','src/pseudo_label.py','src/baselines.py','main.py','run_clean_reproduction.py','run_faga_ablation.py','run_plp_ablation.py']:
    py_compile.compile(str(ROOT/rel), doraise=True)
print('Journal fixes applied; modified Python files compile successfully.')


Journal fixes applied; modified Python files compile successfully.


### Compatibility fixes for GNNExplainer and plotting
These preserve the repository's existing optional explanation and plotting stages.

In [5]:
from pathlib import Path
import py_compile

# GNNExplainer adapter: expose the signature expected by PyG's Explainer API.
p = Path("src/explain.py")
s = p.read_text()
if "class _ExplainWrap" not in s:
    wrapper = "\nclass _ExplainWrap(torch.nn.Module):\n    \"\"\"Adapt ElliGAT.forward(data) to the x/edge_index interface used by Explainer.\"\"\"\n    def __init__(self, model):\n        super().__init__()\n        self.model = model\n    def forward(self, x, edge_index, edge_attr=None):\n        return self.model(Data(x=x, edge_index=edge_index, edge_attr=edge_attr))\n\n"
    marker = "def explain_fraud_nodes("
    assert marker in s
    s = s.replace(marker, wrapper + marker, 1)
    old = "    explainer = Explainer(\n        model=model,"
    assert old in s
    s = s.replace(old, "    explainer = Explainer(\n        model=_ExplainWrap(model),", 1)
    if "from torch_geometric.data import Data" not in s:
        s = "from torch_geometric.data import Data\n" + s
    p.write_text(s)

# Plotting adapter: recursively move tensors to CPU NumPy before Matplotlib.
p = Path("src/visualize.py")
s = p.read_text()
if "def _save_results_figure_impl(" not in s:
    assert "def save_results_figure(" in s
    s = s.replace("def save_results_figure(", "def _save_results_figure_impl(", 1)
    s += "\n\ndef _cpuify(obj):\n    import torch\n    if isinstance(obj, torch.Tensor):\n        return obj.detach().cpu().numpy()\n    if isinstance(obj, dict):\n        return {k: _cpuify(v) for k, v in obj.items()}\n    if isinstance(obj, (list, tuple)):\n        return type(obj)(_cpuify(v) for v in obj)\n    return obj\n\ndef save_results_figure(*args, **kwargs):\n    return _save_results_figure_impl(*_cpuify(args), **_cpuify(kwargs))\n"
    p.write_text(s)
for f in ["src/explain.py", "src/visualize.py"]:
    py_compile.compile(f, doraise=True)
print("Compatibility patches compile.")


Compatibility patches compile.


In [6]:
# Regression checks for validation-only threshold selection and temporal splitting.
import numpy as np, pandas as pd
from src.trainer import compute_metrics, find_best_threshold
from src.data import chronological_split

y_eval = np.array([0, 0, 0, 1], dtype=int)
p_eval = np.array([0.40, 0.40, 0.40, 0.45], dtype=float)
fixed = compute_metrics(y_eval, p_eval)  # default must remain 0.5, not optimize on y_eval
assert fixed["threshold"] == 0.5, fixed["threshold"]
val_threshold = find_best_threshold(y_eval, p_eval)
tuned = compute_metrics(y_eval, p_eval, threshold=val_threshold)
assert tuned["threshold"] == val_threshold
assert tuned["F1"] >= fixed["F1"]

synthetic = pd.DataFrame({
    "timestep": [1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,9,10,10],
    "txId": list(range(20)), "isFraud": [0,1]*10,
})
tr, va, te, ntr, nva = chronological_split(synthetic, 0.6, 0.2)
assert set(tr.timestep).isdisjoint(set(va.timestep))
assert set(va.timestep).isdisjoint(set(te.timestep))
assert tr.timestep.max() < va.timestep.min()
assert va.timestep.max() < te.timestep.min()
print("PASS: compute_metrics never tunes its default threshold on evaluation labels.")
print(f"PASS: validation threshold can be passed explicitly (threshold={val_threshold:.3f}).")
print("PASS: chronological split keeps each timestep in exactly one partition.")


PASS: compute_metrics never tunes its default threshold on evaluation labels.
PASS: validation threshold can be passed explicitly (threshold=0.405).
PASS: chronological split keeps each timestep in exactly one partition.


## 4. Download and verify the Elliptic dataset

In [7]:
os.makedirs(DATA_DIR, exist_ok=True)
for name in ["features", "edgelist", "classes"]:
    url = f"https://data.pyg.org/datasets/elliptic/elliptic_txs_{name}.csv.zip"
    dest = f"{DATA_DIR}/elliptic_txs_{name}.csv.zip"
    if not os.path.exists(dest):
        r = subprocess.run(["wget", "-q", "-O", dest, url])
        if r.returncode:
            raise RuntimeError(f"Dataset download failed: {url}")
    subprocess.run(["unzip", "-o", "-q", dest, "-d", DATA_DIR], check=True)

found = glob.glob(f"{DATA_DIR}/**/elliptic_txs_features.csv", recursive=True)
assert found, "elliptic_txs_features.csv missing after download"
DATA_DIR = os.path.dirname(found[0])
for name in ["features", "classes", "edgelist"]:
    assert os.path.exists(f"{DATA_DIR}/elliptic_txs_{name}.csv"), f"Missing {name} CSV"
print("Dataset ready at:", DATA_DIR)
print("Files:", sorted(os.path.basename(x) for x in glob.glob(f"{DATA_DIR}/elliptic_txs_*.csv")))


Dataset ready at: /content/data/elliptic
Files: ['elliptic_txs_classes.csv', 'elliptic_txs_edgelist.csv', 'elliptic_txs_features.csv']


## 5. Preflight: splits, labels, graph direction, and PLP eligibility

In [8]:
import numpy as np, pandas as pd, torch
from src.data import load_elliptic, add_velocity_features, chronological_split, build_graph, VELOCITY_COLS

os.environ["BITCOIN_DATA_PATH"] = DATA_DIR
os.environ["ELLIPTIC_CACHE_PATH"] = DATA_DIR

df, X_all, edge_index, edge_attr, txid_to_idx, all_nodes = load_elliptic(DATA_DIR)
df = add_velocity_features(df)
train_df, val_df, test_df, train_size, val_size = chronological_split(df)
train_cutoff = int(train_df["timestep"].max())
val_cutoff = int(val_df["timestep"].max())
assert train_df["timestep"].max() < val_df["timestep"].min()
assert val_df["timestep"].max() < test_df["timestep"].min()
assert set(train_df["timestep"]).isdisjoint(set(val_df["timestep"]))
assert set(val_df["timestep"]).isdisjoint(set(test_df["timestep"]))

feat_cols = [c for c in df.columns if c.startswith("f")]
graph_data, scaler_raw, _ = build_graph(
    df, X_all, edge_index, edge_attr, txid_to_idx,
    train_size, val_size, feat_cols, all_nodes=all_nodes
)
# Every graph edge must respect chronological message direction.
src_t = graph_data.node_timestep[graph_data.edge_index[0]]
dst_t = graph_data.node_timestep[graph_data.edge_index[1]]
assert bool(torch.all(src_t <= dst_t)), "Future-to-past graph edge detected"
assert bool(torch.all((~graph_data.plp_eligible_mask) | (graph_data.node_timestep <= train_cutoff)))

report = {
    "total_nodes": int(len(all_nodes)),
    "labelled_nodes": int(len(df)),
    "raw_feature_count": int(X_all.shape[1]),
    "graph_edges": int(graph_data.edge_index.shape[1]),
    "train_rows": int(len(train_df)), "validation_rows": int(len(val_df)), "test_rows": int(len(test_df)),
    "train_timestep_minmax": [int(train_df.timestep.min()), int(train_df.timestep.max())],
    "validation_timestep_minmax": [int(val_df.timestep.min()), int(val_df.timestep.max())],
    "test_timestep_minmax": [int(test_df.timestep.min()), int(test_df.timestep.max())],
    "train_class_counts": {str(int(k)): int(v) for k,v in train_df.isFraud.value_counts().items()},
    "validation_class_counts": {str(int(k)): int(v) for k,v in val_df.isFraud.value_counts().items()},
    "test_class_counts": {str(int(k)): int(v) for k,v in test_df.isFraud.value_counts().items()},
    "plp_eligible_unlabelled_nodes": int(graph_data.plp_eligible_mask.sum()),
    "synthetic_velocity_features_enabled": bool(VELOCITY_COLS),
    "temporal_edge_direction_check": "PASS",
}
print(json.dumps(report, indent=2))
with open(os.path.join(DRIVE_ROOT, "preflight_report.json"), "w") as f:
    json.dump(report, f, indent=2)
del graph_data, scaler_raw, X_all, edge_index, edge_attr, df, all_nodes, txid_to_idx, train_df, val_df, test_df
import gc; gc.collect(); torch.cuda.empty_cache()


  Loaded edges: 234355 directed → 468710 time-respecting directed
  [Elliptic] 203,769 total | 46,564 labelled | Fraud: 0.0976
{
  "total_nodes": 203769,
  "labelled_nodes": 46564,
  "raw_feature_count": 165,
  "graph_edges": 468710,
  "train_rows": 29894,
  "validation_rows": 7829,
  "test_rows": 8841,
  "train_timestep_minmax": [
    1,
    34
  ],
  "validation_timestep_minmax": [
    35,
    41
  ],
  "test_timestep_minmax": [
    42,
    49
  ],
  "train_class_counts": {
    "0": 26432,
    "1": 3462
  },
  "validation_class_counts": {
    "0": 7154,
    "1": 675
  },
  "test_class_counts": {
    "0": 8433,
    "1": 408
  },
  "plp_eligible_unlabelled_nodes": 106371,
  "synthetic_velocity_features_enabled": false,
  "temporal_edge_direction_check": "PASS"
}


## 6. Full corrected experiment — five seeds, resumable

Each seed runs in its own Drive folder. A seed is marked complete only when `main.py` exits successfully. If Colab disconnects, rerun this cell after setup; completed seeds will be skipped. Full runs can take several hours depending on GPU availability.

In [9]:
from IPython.display import display

RUN_ROOT = os.path.join(DRIVE_ROOT, "full_run")
os.makedirs(RUN_ROOT, exist_ok=True)

def run_one_seed(seed):
    seed_dir = os.path.join(RUN_ROOT, f"seed-{seed}")
    done = os.path.join(seed_dir, "DONE.txt")
    if os.path.exists(done):
        print(f"[skip] seed {seed} already completed: {seed_dir}")
        return
    log_path = os.path.join(RUN_ROOT, f"seed-{seed}.log")
    hidden = int(HIDDEN_DIM)
    while True:
        if os.path.isdir(seed_dir):
            shutil.rmtree(seed_dir)
        os.makedirs(seed_dir, exist_ok=True)
        env = os.environ.copy()
        env.update({
            "BITCOIN_DATA_PATH": DATA_DIR,
            "ELLIPTIC_CACHE_PATH": DATA_DIR,
            "OUTPUT_DIR": seed_dir,
            "SEEDS": str(seed),
            "EPOCHS": str(EPOCHS),
            "PRETRAIN_EPOCHS": str(PRETRAIN_EPOCHS),
            "PLP_EPOCHS": str(PLP_EPOCHS),
            "HIDDEN_DIM": str(hidden),
            "LOSS_NAME": LOSS_NAME,
            "EDGE_ENCODING": EDGE_ENCODING,
            "MC_SAMPLES": str(MC_SAMPLES),
            "EXPLAIN_NODES": str(EXPLAIN_NODES),
            "EXPLAIN_EPOCHS": str(EXPLAIN_EPOCHS),
            "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        })
        print(f"\n===== Seed {seed} | hidden_dim={hidden} =====")
        tail = []
        with open(log_path, "a", buffering=1) as log:
            log.write(f"\n===== START seed={seed} hidden_dim={hidden} at {time.ctime()} =====\n")
            proc = subprocess.Popen([sys.executable, "-u", "main.py"], cwd=REPO_DIR,
                                    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                    text=True, bufsize=1)
            for line in proc.stdout:
                print(line, end="")
                log.write(line)
                tail.append(line)
                tail = tail[-250:]
            rc = proc.wait()
            log.write(f"\n===== EXIT code={rc} =====\n")
        if rc == 0 and os.path.exists(os.path.join(seed_dir, "metrics.json")):
            Path(done).write_text(f"completed seed={seed}; hidden_dim={hidden}; time={time.ctime()}\n")
            print(f"[done] seed {seed} saved to Drive")
            return
        tail_text = "".join(tail)
        if ("OutOfMemoryError" in tail_text or "CUDA out of memory" in tail_text) and hidden > 32:
            hidden //= 2
            print(f"[OOM] retrying seed {seed} with hidden_dim={hidden}")
            gc.collect(); torch.cuda.empty_cache()
            continue
        raise RuntimeError(f"Seed {seed} failed with exit code {rc}. Inspect Drive log: {log_path}")

for seed in SEEDS:
    run_one_seed(seed)
print("All configured seeds have completed.")



===== Seed 42 | hidden_dim=128 =====
2026-10-21 16:17:39 | INFO    | Novelty: ElliGAT-ASL loss=asymmetric gamma_pos=0.0 gamma_neg=4.0

  Bitcoin Fraud Detection — ElliGAT
  Device : cuda
  Output : /content/drive/MyDrive/ElliGAT_Journal_Corrected/full_run/seed-42

Step 1: Loading Elliptic Bitcoin dataset …
  [check] found dataset at: /content/data/elliptic
  Path: /content/data/elliptic
  Loaded edges: 234355 directed → 468710 time-respecting directed
  [Elliptic] 203,769 total | 46,564 labelled | Fraud: 0.0976

Step 2: Excluding synthetic proxies; using original Elliptic features …

Step 3: Chronological train/val/test split (70/15/15) …
  Train: 29,894  Val: 7,829  Test: 8,841
  Tabular features: 165 (7 velocity + 165 raw)
  Class balance — Fraud: 3462 (11.581%)

Step 4: Building Bitcoin transaction graph …
  Nodes: 203,769  Edges: 468,710  Features/node: 165
  Methods to run: ['ElliGAT', 'EvolveGCN', 'BaselineGNN', 'ROLAND', 'TGN', 'Tabular', 'Meta']

Step 5: Multi-seed experiments

## 7. Aggregate corrected results and inspect per-seed metrics

In [10]:
# Aggregate the per-seed metrics.json files, then build a simple per-seed table.
AGG_DIR = os.path.join(RUN_ROOT, "aggregate")
os.makedirs(AGG_DIR, exist_ok=True)
subprocess.run([sys.executable, "scripts/aggregate_results.py", RUN_ROOT, AGG_DIR], cwd=REPO_DIR, check=False)

seed_rows=[]
for seed in SEEDS:
    mp=os.path.join(RUN_ROOT, f"seed-{seed}", "metrics.json")
    if not os.path.exists(mp):
        print("Missing metrics:", mp); continue
    m=json.load(open(mp))
    detailed = m.get("_per_seed_metrics", {})
    if detailed:
        for model, run_list in detailed.items():
            for vals in run_list:
                row={"seed":seed,"model":model}
                row.update({k:v for k,v in vals.items() if isinstance(v,(int,float))})
                seed_rows.append(row)
        # PLP is a post-hoc single-seed result in each seed-specific run.
        vals=m.get("ElliGAT+PLP", {})
        if isinstance(vals,dict):
            row={"seed":seed,"model":"ElliGAT+PLP"}
            for metric in ["ROC-AUC","PR-AUC","F1","MCC","Balanced Acc","Precision","Recall","AURC","threshold","TN","FP","FN","TP"]:
                value=vals.get(metric)
                if isinstance(value,dict) and "mean" in value: row[metric]=value["mean"]
                elif isinstance(value,(int,float)): row[metric]=value
            if len(row)>2: seed_rows.append(row)
    else:
        for model, vals in m.items():
            if model.startswith("_") or not isinstance(vals, dict): continue
            row={"seed":seed,"model":model}
            for metric in ["ROC-AUC","PR-AUC","F1","MCC","Balanced Acc","Precision","Recall","AURC","threshold","TN","FP","FN","TP"]:
                if metric in vals and isinstance(vals[metric], (int,float)):
                    row[metric]=vals[metric]
                elif metric in vals and isinstance(vals[metric], dict) and "mean" in vals[metric]:
                    row[metric]=vals[metric]["mean"]
            seed_rows.append(row)
per_seed=pd.DataFrame(seed_rows)
if not per_seed.empty:
    display(per_seed.sort_values(["model","seed"]))
    numeric=[c for c in ["ROC-AUC","PR-AUC","F1","MCC","Balanced Acc","Precision","Recall","AURC"] if c in per_seed.columns]
    summary=per_seed.groupby("model")[numeric].agg(["mean","std"]).round(4)
    display(summary)
    per_seed.to_csv(os.path.join(AGG_DIR,"corrected_per_seed_metrics.csv"),index=False)
    summary.to_csv(os.path.join(AGG_DIR,"corrected_mean_std_summary.csv"))
else:
    print("No metrics found yet; run the full experiment cell first.")


,seed,model,ROC-AUC,PR-AUC,TN,FP,FN,TP,F1,MCC,...,AUGT/Precision,AUGT/Recall,AUGT/Deferred_Fraud,AUGT/Deferred_Licit,AURC,AUGT/base_low,AUGT/base_high,AUGT/k,threshold,Accuracy
35,17,BaselineGNN,0.468685,0.040119,4044.0,4389.0,205.0,203.0,0.081200,-0.009619,...,0.045006,1.0,130.0,2534.0,0.958936,0.05,0.646552,0.355556,0.940,NaN
2,42,BaselineGNN,0.448048,0.038157,3559.0,4874.0,187.0,221.0,0.080320,-0.015413,...,0.042663,1.0,144.0,2509.0,0.963298,0.05,0.687931,0.311111,0.945,NaN
46,99,BaselineGNN,0.484514,0.042879,3772.0,4661.0,190.0,218.0,0.082466,-0.007761,...,0.042351,1.0,158.0,2780.0,0.953270,0.05,0.660345,0.400000,0.950,NaN
13,123,BaselineGNN,0.465681,0.040166,3934.0,4499.0,195.0,213.0,0.083203,-0.004811,...,0.041490,1.0,154.0,2565.0,0.958060,0.05,0.715517,0.222222,0.950,NaN
24,2024,BaselineGNN,0.510648,0.044683,4290.0,4143.0,199.0,209.0,0.087815,0.008801,...,0.046398,1.0,122.0,2555.0,0.953683,0.05,0.660345,0.355556,0.950,NaN
33,17,ElliGAT,0.520872,0.046673,4719.0,3714.0,203.0,205.0,0.094754,0.026201,...,0.047766,1.0,113.0,2552.0,0.949719,0.05,0.632759,0.355556,0.915,NaN
0,42,ElliGAT,0.572772,0.060487,5120.0,3313.0,201.0,207.0,0.105397,0.049071,...,0.051574,1.0,82.0,2438.0,0.941162,0.05,0.605172,0.266667,0.890,NaN
44,99,ElliGAT,0.560097,0.058238,4925.0,3508.0,192.0,216.0,0.104550,0.048198,...,0.050560,1.0,83.0,2330.0,0.936955,0.05,0.563793,0.400000,0.910,NaN
11,123,ElliGAT,0.583299,0.073851,6588.0,1845.0,253.0,155.0,0.128738,0.080796,...,0.051555,1.0,103.0,2822.0,0.915936,0.05,0.605172,0.400000,0.930,NaN
22,2024,ElliGAT,0.559570,0.070961,5689.0,2744.0,199.0,209.0,0.124368,0.083126,...,0.049896,1.0,72.0,2035.0,0.952650,0.05,0.550000,0.311111,0.810,NaN


ROC-AUC          PR-AUC              F1             MCC          \
                mean     std    mean     std    mean     std    mean     std   
model                                                                          
BaselineGNN   0.4755  0.0235  0.0412  0.0026  0.0830  0.0029 -0.0058  0.0090   
ElliGAT       0.5593  0.0236  0.0620  0.0109  0.1116  0.0144  0.0575  0.0242   
ElliGAT+PLP   0.5615  0.0259  0.0627  0.0112  0.1107  0.0144  0.0564  0.0228   
EvolveGCN     0.5071  0.0496  0.0432  0.0052  0.0971  0.0079  0.0527  0.0161   
LightGBM      0.8531  0.0054  0.5543  0.0020  0.6085  0.0041  0.6221  0.0076   
MLP           0.8769  0.0052  0.4768  0.0287  0.5259  0.0156  0.5203  0.0145   
MetaEnsemble  0.8609  0.0035  0.5394  0.0034  0.6241  0.0025  0.6457  0.0048   
ROLAND        0.7235  0.0196  0.1180  0.0222  0.1433  0.0736  0.1139  0.0626   
RandomForest  0.8612  0.0045  0.5452  0.0014  0.5883  0.0036  0.5938  0.0062   
TGN           0.7272  0.0167  0.1603  0.0141  0.2412  0.0172  0.2033  0.0200   
XGBoost       0.8514  0.0280  0.5498  0.0135  0.6046  0.0149  0.6179  0.0212   

             Balanced Acc         Precision          Recall            AURC  \
                     mean     std      mean     std    mean     std    mean   
model                                                                         
BaselineGNN        0.4932  0.0107    0.0451  0.0018  0.5216  0.0175  0.9574   
ElliGAT            0.5638  0.0241    0.0635  0.0103  0.4863  0.0603  0.9393   
ElliGAT+PLP        0.5630  0.0216    0.0628  0.0104  0.4946  0.0588  0.9380   
EvolveGCN          0.5436  0.0291    0.0513  0.0047  0.9279  0.0843  0.9292   
LightGBM           0.7354  0.0017    0.8470  0.0238  0.4750  0.0040     NaN   
MLP                0.7125  0.0103    0.6643  0.0234  0.4358  0.0213     NaN   
MetaEnsemble       0.7370  0.0005    0.9044  0.0146  0.4765  0.0013     NaN   
ROLAND             0.5556  0.0433    0.1686  0.0427  0.1456  0.1027  0.8312   
RandomForest       0.7322  0.0013    0.7847  0.0182  0.4706  0.0030     NaN   
TGN                0.6318  0.0192    0.1912  0.0189  0.3324  0.0490  0.8033   
XGBoost            0.7338  0.0023    0.8418  0.0486  0.4721  0.0033     NaN   

                      
                 std  
model                 
BaselineGNN   0.0041  
ElliGAT       0.0145  
ElliGAT+PLP   0.0145  
EvolveGCN     0.0048  
LightGBM         NaN  
MLP              NaN  
MetaEnsemble     NaN  
ROLAND        0.0422  
RandomForest     NaN  
TGN           0.0164  
XGBoost          NaN

## 9. Save a reproducibility bundle to Drive and download it

In [17]:
# Include corrected code, protocol notes, per-seed metrics, logs, and plots.
BUNDLE_PATH=os.path.join(DRIVE_ROOT,"elligat_journal_corrected_bundle.zip")
TEMP_BUNDLE="/content/elligat_journal_corrected_bundle.zip"
if os.path.exists(TEMP_BUNDLE): os.remove(TEMP_BUNDLE)
shutil.make_archive(TEMP_BUNDLE[:-4], "zip", DRIVE_ROOT)
shutil.copy2(TEMP_BUNDLE, BUNDLE_PATH)
print("Saved bundle:", BUNDLE_PATH)
print("Drive root contents:", os.listdir(DRIVE_ROOT))
from google.colab import files
files.download(TEMP_BUNDLE)


Saved bundle: /content/drive/MyDrive/ElliGAT_Journal_Corrected/elligat_journal_corrected_bundle.zip
Drive root contents: ['preflight_report.json', 'full_run', 'elligat_journal_corrected_bundle.zip', 'faga_ablation', 'plp_ablation']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>